# Sign head and moment fidelity vs ADC bit depth (no training)

**Why this runs first.** The trained bit-depth ablation is ~45 min per point. Before spending
that, measure directly how much of the ansatz survives quantization. Everything here is a
closed-form observable on digitized clusters — no model, no fitting beyond a 1-feature logistic.

**The thing most at risk.** The sign of both angles comes from a *between-slice centroid drift*:
`Tx` (columns, 50 um) signs cot a, `Ty` (rows, 12.5 um) signs cot b. The fitted `a0_beta` is
about **-7.5 um on a 12.5 um pitch** — well under one pixel. That is a small difference of two
nearly equal centroids, and it is exactly the kind of quantity coarse quantization destroys,
while the cluster *widths* that set the angle magnitudes barely notice. So expect the sign to
degrade first and fastest.

**What gets measured, per bit depth:**

| column | meaning |
|---|---|
| `signA/signB` | balanced accuracy of the single-scalar drift as a sign predictor |
| `ceilA/ceilB` | balanced accuracy of logistic regression on all 512 raw digitized pixels |
| `corr x/y` | correlation of the col/row centroid with the position label |
| `corr cotA/cotB` | correlation of (col/row width - 1) with the angle magnitude |
| `med\|Tx\|/\|Ty\|` | median drift magnitude in um — watch this collapse toward 0 |

`ceil` vs `sign` is the important pair. If the scalar drops but the ceiling holds, the sign
information is still in the cluster and a better observable exists. If the ceiling drops too,
the information is genuinely gone and no amount of model capacity recovers it.

Two level conventions are compared: `code` (levels 0..2^n-1, what an ASIC emits) and `midpoint`
(levels at the charge center of each bin). The gap between them separates *lost dynamic range*
from *coarse weighting*.

Run 1 -> 2 -> 3. Cell 3 needs only `tg`, `vg` from cell 2.

In [1]:
# ---- 1. setup ----
import os
WORKDIR = "/depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-digitization"
HELPERS = os.path.join(WORKDIR, "two_bit_optimization_helpers")
os.chdir(WORKDIR)

import sys, json
sys.path.insert(0, HELPERS)

import numpy as np
import tensorflow as tf

for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

from prepare_tfrecords import generate_tfrecords, load_tfrecords
from symbolic.digitize import thresholds_from_data, levels_for

print("TF", tf.__version__, "| workdir:", os.getcwd())

2026-07-30 18:49:49.068939: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-30 18:49:49.069007: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-30 18:49:49.070174: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-30 18:49:49.077466: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-30 18:49:49.785359: W tensorflow/compiler/tf2

TF 2.15.1 | workdir: /depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-digitization


In [2]:
# ---- 2. data (analog, clean -- same config as the training notebook) ----
import os, json, numpy as np
from prepare_tfrecords import generate_tfrecords, load_tfrecords

SEED             = 42
DATASET_DIR      = "/depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets"
SELECT_CONTAINED = True
TIMESLICES       = 2
BATCH            = 5000
TFRECORDS_EXIST  = True

_, _, tfr_tr, tfr_val = generate_tfrecords(
    dataset_dir=DATASET_DIR, model_type="ViT_Max",
    train_batch_size=BATCH, val_batch_size=BATCH,
    select_contained=SELECT_CONTAINED, timeslices=TIMESLICES,
    tfrecords_exist=TFRECORDS_EXIST, seed=SEED,
)
# NOTE: digitize=False on purpose -- this notebook does its own quantization so
# it can sweep bit depth. Never stack the generator quantizer on top of it.
tg, vg = load_tfrecords(tfr_tr, tfr_val, noise=-1, digitize=False, seed=SEED)

labels_scale = json.load(open(os.path.join(tfr_tr, "metadata.json")))["labels_scale"]
print("labels_scale:", np.round(labels_scale, 3), "| train batches:", len(tg), "| val batches:", len(vg))

Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_train_contained/metadata.json


Loading metadata from /depot/cms/users/kuang14/Smart_Pixel/dataset_s_series/dataset_3sr/dataset_3sr_16x16_50x12P5_centeredIncidence_parquets/TFR_files/2t/TFR_test_contained/metadata.json


labels_scale: [123.41   30.93    6.577   1.93 ] | train batches: 87 | val batches: 22


In [3]:
# ---- 3. THE STUDY ----  needs only `tg`, `vg`, `labels_scale` from cell 2
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from symbolic.digitize import thresholds_from_data, levels_for

N_VAL_BATCHES = 10        # 10 x 5000 = 50k events for the measurements
N_THR_BATCHES = 4         # train batches used only to place the thresholds
OFFSET        = 80.0      # e-, below this reads level 0 (matches two_bit_optimization.py)
SCHEMES       = ["quantile", "lowfirst"]   # placement matters more than depth for |cot b|
DO_CEILING    = True      # 512-pixel logistic ceiling; set False if too slow
P_X, P_Y      = 50.0, 12.5

def collect(gen, n):
    n = min(n, len(gen))
    X, Y = zip(*[(np.asarray(gen[i][0], "float32"), np.asarray(gen[i][1], "float32")) for i in range(n)])
    return np.concatenate(X), np.concatenate(Y)

Xt, _  = collect(tg, N_THR_BATCHES)
Xv, Yv = collect(vg, N_VAL_BATCHES)
Yp = Yv * np.asarray(labels_scale, "float32")
lab_x, lab_y, cotA, cotB = Yp[:, 0], Yp[:, 1], Yp[:, 2], Yp[:, 3]

nz = Xt[Xt > OFFSET]
print(f"events {len(Xv)} | above-offset pixels {(Xt > OFFSET).mean():.1%} | "
      f"charge median {np.median(nz):.0f}  p99.5 {np.quantile(nz, 0.995):.0f} e-")

# ---------------- observables (v3 axis convention: cols=x@50um, rows=y@12.5um) ----------------
idx = np.arange(16, dtype=np.float32)

def observables(q4):
    """q4 (N,16,16,2) -> centroids, widths and between-slice drifts, all in um."""
    q = np.maximum(q4, 0.0)
    qs = q.sum(-1)
    px = qs.sum(1); py = qs.sum(2)                    # col-indexed (x), row-indexed (y)
    cx = (px * idx).sum(1) / (px.sum(1) + 1e-9) * P_X
    cy = (py * idx).sum(1) / (py.sum(1) + 1e-9) * P_Y
    wx = (px > 0).sum(1).astype(np.float32)
    wy = (py > 0).sum(1).astype(np.float32)
    # charge-weighted RMS width: uses the surviving pixels' VALUES, so unlike the
    # hard `prof > 0` count it is not pinned to the zero-suppression threshold.
    def rms(p):
        w = p / (p.sum(1, keepdims=True) + 1e-9)
        c = (w * idx).sum(1, keepdims=True)
        return np.sqrt((w * (idx - c) ** 2).sum(1) + 1e-9)
    rx, ry = rms(px), rms(py)
    def drift(a):
        p0, p1 = q[..., 0].sum(a), q[..., -1].sum(a)
        c0 = (p0 * idx).sum(1) / (p0.sum(1) + 1e-9)
        c1 = (p1 * idx).sum(1) / (p1.sum(1) + 1e-9)
        return c1 - c0
    Tx = drift(1) * P_X                               # collapse rows -> x drift
    Ty = drift(2) * P_Y                               # collapse cols -> y drift
    return cx, cy, wx, wy, rx, ry, np.nan_to_num(Tx), np.nan_to_num(Ty)

def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (np.sqrt((a * a).sum() * (b * b).sum()) + 1e-12))

def bal_acc(feat, target_pos, max_iter=400):
    f = np.asarray(feat, "float32").reshape(len(feat), -1)
    # standardize: the 512-pixel probe does not converge on raw charge, and an
    # unconverged probe reads as a low "ceiling" and invites the wrong conclusion.
    f = (f - f.mean(0)) / (f.std(0) + 1e-6)
    clf = LogisticRegression(max_iter=max_iter).fit(f, target_pos)
    return balanced_accuracy_score(target_pos, clf.predict(f))

def digitize(x, thr, lv):
    return lv[np.clip(np.digitize(x, thr), 0, len(lv) - 1)].astype(np.float32)

# ---------------- sweep ----------------
sA, sB = (cotA > 0).astype(int), (cotB > 0).astype(int)
settings = ([(None, None, "analog")]
            + [(b, s, m) for b in (4, 3, 2) for s in SCHEMES for m in ("code", "midpoint")])

hdr = (f"{'bits':>7} {'scheme':>9} {'levels':>9} | {'signA':>6} {'signB':>6} | {'ceilA':>6} {'ceilB':>6} | "
       f"{'corr x':>7} {'corr y':>7} | {'c|cotA|':>8} {'c|cotB|':>8} | {'rms cotA':>8} {'rms cotB':>8}")
print("\n" + "=" * len(hdr)); print(hdr); print("=" * len(hdr))

rows = []
for nbits, scheme, mode in settings:
    if nbits is None:
        Xd, tag = Xv, ("analog", "--", "--")
    else:
        thr, edges = thresholds_from_data(Xt, nbits, offset=OFFSET, scheme=scheme)
        lv = levels_for(nbits, mode, edges)
        Xd, tag = digitize(Xv, thr, lv), (f"{nbits}-bit", scheme, mode)

    cx, cy, wx, wy, rx, ry, Tx, Ty = observables(Xd)
    r = dict(bits=tag[0], scheme=tag[1], levels=tag[2],
             signA=bal_acc(Tx[:, None], sA), signB=bal_acc(Ty[:, None], sB),
             corr_x=corr(cx, lab_x), corr_y=corr(cy, lab_y),
             corr_cotA=corr(wx - 1, np.abs(cotA)), corr_cotB=corr(wy - 1, np.abs(cotB)),
             rms_cotA=corr(rx, np.abs(cotA)), rms_cotB=corr(ry, np.abs(cotB)),
             mTx=float(np.median(np.abs(Tx))), mTy=float(np.median(np.abs(Ty))))
    if DO_CEILING:
        flat = np.maximum(Xd, 0.0).reshape(len(Xd), -1)
        r["ceilA"] = bal_acc(flat, sA, max_iter=3000)
        r["ceilB"] = bal_acc(flat, sB, max_iter=3000)
    else:
        r["ceilA"] = r["ceilB"] = float("nan")
    rows.append(r)
    print(f"{r['bits']:>7} {r['scheme']:>9} {r['levels']:>9} | {r['signA']:6.4f} {r['signB']:6.4f} | "
          f"{r['ceilA']:6.4f} {r['ceilB']:6.4f} | {r['corr_x']:7.3f} {r['corr_y']:7.3f} | "
          f"{r['corr_cotA']:8.3f} {r['corr_cotB']:8.3f} | {r['rms_cotA']:8.3f} {r['rms_cotB']:8.3f}")

# ---------------- verdict ----------------
a = rows[0]
def pick(bits, scheme, mode="code"):
    m = [r for r in rows if r["bits"] == bits and r["scheme"] == scheme and r["levels"] == mode]
    return m[0] if m else None
two = pick("2-bit", "quantile")
print("\n" + "=" * 74)
print("VERDICT  (analog -> 2-bit code)")
print("=" * 74)
for k, nm in (("signA", "sign(cot a)"), ("signB", "sign(cot b)"),
              ("corr_x", "corr x"), ("corr_y", "corr y"),
              ("corr_cotA", "corr |cot a|"), ("corr_cotB", "corr |cot b|")):
    d = two[k] - a[k]
    print(f"  {nm:14s} {a[k]:7.4f} -> {two[k]:7.4f}   ({d:+.4f})")
low = pick("2-bit", "lowfirst")
if low is not None:
    print("\n  threshold placement at 2 bits (quantile -> lowfirst):")
    for k, nm in (("corr_cotB", "corr |cot b|"), ("corr_cotA", "corr |cot a|"),
                  ("signB", "sign(cot b)"), ("signA", "sign(cot a)")):
        print(f"    {nm:14s} {two[k]:7.4f} -> {low[k]:7.4f}   ({low[k] - two[k]:+.4f})")
    print(f"\n  hard count vs charge-weighted RMS width (2-bit, best scheme):")
    best = max((r for r in rows if r["bits"] == "2-bit"), key=lambda r: r["rms_cotB"])
    print(f"    |cot b|:  count {best['corr_cotB']:.4f}  ->  RMS {best['rms_cotB']:.4f}"
          f"   (analog count {a['corr_cotB']:.4f}, analog RMS {a['rms_cotB']:.4f})")
    if best["rms_cotB"] > best["corr_cotB"] + 0.02:
        print("    -> the RMS width survives zero-suppression better; worth changing cotb_abs.")
    else:
        print("    -> no gain from the RMS width; keep the hard count.")
    rec = low["corr_cotB"] - two["corr_cotB"]
    lost = a["corr_cotB"] - two["corr_cotB"]
    print(f"    -> lowfirst recovers {rec / lost:.0%} of the |cot b| loss"
          if lost > 1e-6 else "    -> no |cot b| loss to recover")

if DO_CEILING:
    # This is a linear probe on 512 standardized pixels, NOT an information bound.
    # A probe scoring BELOW the physics scalar means the probe is the weaker model.
    gapA, gapB = two["ceilA"] - two["signA"], two["ceilB"] - two["signB"]
    print(f"\n  512-pixel linear probe minus single scalar: cot a {gapA:+.4f}, cot b {gapB:+.4f}")
    if max(gapA, gapB) > 0.02:
        print("  -> the raw cluster still knows more than the scalar; a better observable may exist.")
    elif min(gapA, gapB) < -0.005:
        print("  -> the scalar BEATS the linear probe: the probe is underpowered, not a ceiling.")
    else:
        print("  -> scalar and linear probe agree; no cheap gain from a richer linear observable.")
json.dump(rows, open("sign_vs_bitdepth.json", "w"), indent=1)
print("\nwrote sign_vs_bitdepth.json")

2026-07-30 18:49:52.583207: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2909 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 1g.5gb, pci bus id: 0000:81:00.0, compute capability: 8.0


events 50000 | above-offset pixels 6.0% | charge median 420  p99.5 10418 e-

   bits    scheme    levels |  signA  signB |  ceilA  ceilB |  corr x  corr y |  c|cotA|  c|cotB| |  med|Tx|  med|Ty|
 analog        --        -- | 0.9661 0.9644 | 0.9674 0.9690 |   0.927   0.914 |    0.990    0.944 |    54.25    13.40
  4-bit  quantile      code | 0.9661 0.9379 | 0.9546 0.9591 |   0.957   0.934 |    0.992    0.820 |    42.94    12.77
  4-bit  quantile  midpoint | 0.9750 0.9652 | 0.9679 0.9648 |   0.919   0.902 |    0.992    0.820 |    67.07    16.89
  4-bit  lowfirst      code | 0.9666 0.9385 | 0.9547 0.9593 |   0.956   0.933 |    0.992    0.819 |    43.33    12.85
  4-bit  lowfirst  midpoint | 0.9751 0.9653 | 0.9680 0.9647 |   0.919   0.902 |    0.992    0.819 |    67.32    16.94
  3-bit  quantile      code | 0.9680 0.9417 | 0.9552 0.9582 |   0.952   0.931 |    0.992    0.805 |    46.83    13.41
  3-bit  quantile  midpoint | 0.9750 0.9648 | 0.9646 0.9589 |   0.920   0.898 |    0.992    0.805